# Built environment and car ownership

**RQ:** How strongly is mixed-use development associated with reduced car dependence across major U.S. metropolitan city areas, and how does this relationship compare with other characteristics of the built environment, such as population density, street connectivity, and public transportation accessibility?

**Data**
- EPA Smart Location Database v3.0, cleaned in `src/build_sld.py`
- ACS 2014-18 income and commute mode, joined in `src/build_acs.py`
- 1 row = 1 Census block group

**Sample**
- metros with 1M+ people ("major cities")
- block groups with 50+ households

**Outcome**
- `pct_2plus_cars` = % of households with 2+ cars
- averages weighted by # of households

In [1]:
import numpy as np
import pandas as pd
import altair as alt

sld = pd.read_csv("../data/processed/sld_blockgroups.csv.gz", dtype={"geoid": str, "cbsa": str})

# keep major metros + block groups with enough households
df = sld[sld["major_metro"] & sld["enough_households"]].copy()

print(len(df), "block groups")
print(df["cbsa"].nunique(), "metros")
df[["pct_2plus_cars", "mixed_use", "pop_density", "intersection_density", "transit_dist_m"]].describe().round(2)

114860 block groups
54 metros


,pct_2plus_cars,mixed_use,pop_density,intersection_density,transit_dist_m
count,114860.00,114860.00,114860.00,114860.00,72504.00
mean,55.27,0.48,16.68,101.39,444.36
std,23.63,0.23,36.76,83.84,275.91
min,0.00,0.00,0.00,0.00,0.00
25%,38.98,0.31,3.52,44.00,237.38
50%,58.46,0.50,7.95,86.96,387.58
75%,74.23,0.66,15.58,138.48,607.53
max,100.00,1.00,3133.58,1930.42,1207.00


## Step 1 · Put features on the same scale

- 4 features, 4 different units → can't compare directly
- fix: turn each into a **percentile** (0 = most car-oriented, 100 = most urban)
- split into 10 bins → avg % with 2+ cars in each bin
- steeper line = stronger link to car ownership

Transit note
- SLD has no distance if nearest stop > ¾ mile
- those all tied at "worst access" → end up as one point

In [2]:
# transit: closer = better, so flip the sign
df["transit_access"] = -df["transit_dist_m"]
# no stop within 3/4 mile = worst possible
df.loc[df["no_stop_within_3_4mi"], "transit_access"] = -np.inf

# column name -> chart label
FEATURES = {
    "mixed_use": "Mixed use",
    "pop_density": "Population density",
    "intersection_density": "Street connectivity",
    "transit_access": "Transit access",
}
OUTCOME = "pct_2plus_cars"


def profile(data, col, label, outcome=OUTCOME, weight="households", rank_within=None):
    """Avg outcome (weighted) in each of 10 percentile bins of `col`.

    rank_within: None = rank across all block groups (national);
                 ["cbsa"] = rank inside each metro (used in Step 8)
    """
    keep = [col, outcome, weight] + (rank_within or [])
    d = data[keep].dropna().copy()

    # percentile (0-100) and bin (1-10)
    if rank_within:
        d["pct"] = d.groupby(rank_within)[col].rank(pct=True) * 100
    else:
        d["pct"] = d[col].rank(pct=True) * 100
    d["bin"] = np.ceil(d["pct"] / 10).clip(1, 10).astype(int)

    # weighted avg = sum(outcome * weight) / sum(weight)
    d["y_x_w"] = d[outcome] * d[weight]
    g = d.groupby("bin").agg(
        x=("pct", "mean"),
        y_x_w=("y_x_w", "sum"),
        w=(weight, "sum"),
        n=(weight, "size"),
        q25=(outcome, lambda s: s.quantile(0.25)),  # middle 50% of block groups in the bin
        q75=(outcome, lambda s: s.quantile(0.75)),
    ).reset_index()
    g["y"] = (g["y_x_w"] / g["w"]).round(1)
    g["feature"] = label
    return g[["feature", "bin", "x", "y", "q25", "q75", "n"]]


# one profile per feature, stacked into one table
tables = []
for col, label in FEATURES.items():
    tables.append(profile(df, col, label))
prof = pd.concat(tables, ignore_index=True)
prof.round(1)

,feature,bin,x,y,q25,q75,n
0,Mixed use,1,5.0,50.0,31.2,68.1,11486
1,Mixed use,2,15.0,57.9,40.1,77.1,11486
2,Mixed use,3,25.0,59.6,42.8,77.8,11486
3,Mixed use,4,35.0,59.2,42.0,77.5,11486
4,Mixed use,5,45.0,57.8,40.3,76.7,11486
5,Mixed use,6,55.0,56.7,39.8,75.0,11486
6,Mixed use,7,65.0,55.9,39.5,73.6,11486
7,Mixed use,8,75.0,55.5,39.3,73.0,11486
8,Mixed use,9,85.0,54.4,38.9,71.8,11486
9,Mixed use,10,95.0,52.0,38.4,68.9,11486


## Step 2 · Figure 1: which feature tracks car ownership most?

- x = percentile on each feature
- y = % of households with 2+ cars
- 1 line per feature

Design choices
- **direct labels** at the end of each line instead of a legend → no back-and-forth between legend and lines
- mixed use is drawn **thicker** because it is the feature our question is about
- transit's first point is a tie, not a bin: 37% of block groups have no stop within ¾ mile, so they all share one percentile. It's annotated so readers don't mistake it for a gap in the data

In [3]:
FEATURE_ORDER = ["Mixed use", "Population density", "Street connectivity", "Transit access"]
FEATURE_COLORS = ["#d1495b", "#3a6ea5", "#2a9d8f", "#8d6cab"]   # mixed use = red since it's our focus
colors = alt.Scale(domain=FEATURE_ORDER, range=FEATURE_COLORS)

n_bg = len(df)
n_metros = df["cbsa"].nunique()
no_stop_share = df["no_stop_within_3_4mi"].mean()

base = alt.Chart(prof).encode(
    x=alt.X("x:Q", title="Percentile (0 = most car-oriented, 100 = most urban)",
            scale=alt.Scale(domain=[0, 100])),
    y=alt.Y("y:Q", title="Households with 2+ cars (%)", scale=alt.Scale(domain=[15, 80])),
    color=alt.Color("feature:N", scale=colors, legend=None),   # direct labels replace the legend
)

lines = base.mark_line(point=True).encode(
    # thicker line for mixed use
    strokeWidth=alt.condition(alt.datum.feature == "Mixed use", alt.value(4), alt.value(2)),
    tooltip=["feature", alt.Tooltip("y:Q", title="% with 2+ cars"), alt.Tooltip("n:Q", format=",")],
)

# label each line at its last (most urban) point
end_labels = base.transform_filter(alt.datum.bin == 10).mark_text(
    align="left", dx=8, fontSize=12, fontWeight="bold"
).encode(text="feature:N")

# annotate transit's tied "no stop nearby" point
no_stop = prof[(prof["feature"] == "Transit access")].sort_values("bin").head(1).copy()
no_stop["note"] = f"No stop within ¾ mi ({no_stop_share:.0%} of block groups, all tied)"
note = alt.Chart(no_stop).mark_text(align="left", dx=10, dy=-10, fontSize=11, color="#8d6cab").encode(
    x="x:Q", y="y:Q", text="note:N"
)

fig1 = (lines + end_labels + note).properties(
    width=520,
    height=340,
    title=alt.Title(
        "Which neighborhood features track car ownership most closely?",
        subtitle=[f"Household-weighted mean in each tenth of each feature. {n_bg:,} block groups in {n_metros} metros of 1M+ people.",
                  "Source: EPA Smart Location Database v3.0 (car ownership from ACS 2014-18)."],
    ),
)

fig1.save("../figures/fig1_feature_comparison.png", scale_factor=2)
fig1.save("../figures/fig1_feature_comparison.html")
fig1

ValueError: Saving charts in 'png' format requires the vl-convert-python package: see https://altair-viz.github.io/user_guide/saving_charts.html#png-svg-and-pdf-format

**Numbers for write-up**
- `gap` = drop in % with 2+ cars, least urban bin → most urban bin
- `spearman` = rank correlation w/ % with 2+ cars (more negative = stronger)

In [20]:
# first bin = least urban, last bin = most urban
summary = prof.sort_values("bin").groupby("feature").agg(
    least_urban=("y", "first"),
    most_urban=("y", "last"),
)
summary["gap"] = summary["least_urban"] - summary["most_urban"]

for col, label in FEATURES.items():
    summary.loc[label, "spearman"] = df[col].corr(df[OUTCOME], method="spearman")

summary.sort_values("gap", ascending=False).round(2)

,least_urban,most_urban,gap,spearman
feature,,,,
Population density,73.3,21.5,51.8,-0.48
Transit access,68.3,32.2,36.1,-0.52
Street connectivity,68.6,37.4,31.2,-0.36
Mixed use,50.0,52.0,-2.0,-0.02


## Step 3 · Figure 2: does mixed use still matter at similar densities?

- problem: dense places are also mixed, connected, near transit
- → Fig 1 can't separate mixed use from density
- fix: split into 3 density groups (low / medium / high)
- redo the profiles **inside** each group
- line still steep → feature matters even at similar density
- line flat → Fig 1 slope was mostly just density

In [21]:
# 3 equal-size density groups
df["density_group"] = pd.qcut(df["pop_density"], 3, labels=["Low density", "Medium density", "High density"])

tables = []
for group_name in ["Low density", "Medium density", "High density"]:
    group = df[df["density_group"] == group_name]
    for col, label in FEATURES.items():
        if col == "pop_density":
            continue  # skip density itself
        t = profile(group, col, label)
        t["density_group"] = group_name
        tables.append(t)
prof2 = pd.concat(tables, ignore_index=True)

fig2 = alt.Chart(prof2).mark_line(point=True).encode(
    x=alt.X("x:Q", title="Percentile within density group"),
    y=alt.Y("y:Q", title="Households with 2+ cars (%)", scale=alt.Scale(zero=False)),
    color=alt.Color("density_group:N", title="Density group",
                    sort=["Low density", "Medium density", "High density"],
                    scale=alt.Scale(range=["#9ecae1", "#4292c6", "#08306b"])),  # light -> dark blue
    column=alt.Column("feature:N", title=None),
    tooltip=["feature", "density_group", "y", "n"],
).properties(
    width=200,
    height=240,
    title="Holding density fixed, which features still track car ownership?",
)

fig2.save("../figures/fig2_within_density.png", scale_factor=2)
fig2.save("../figures/fig2_within_density.html")
fig2

alt.Chart(...)

In [22]:
# gap (least urban - most urban bin) for each feature x density group
gaps = prof2.sort_values("bin").groupby(["feature", "density_group"]).agg(
    least_urban=("y", "first"),
    most_urban=("y", "last"),
)
gaps["gap"] = gaps["least_urban"] - gaps["most_urban"]

# rows = feature, columns = density group
gaps.reset_index().pivot(index="feature", columns="density_group", values="gap")[
    ["Low density", "Medium density", "High density"]
].round(1)

density_group,Low density,Medium density,High density
feature,,,
Mixed use,7.8,6.5,5.8
Street connectivity,25.1,11.8,-4.1
Transit access,20.7,19.1,29.8


## Step 4 · Add household income

- Fig 2 holds density fixed, but not income
- income matters: richer households own more cars, and income varies with density, transit, and mixed use
- → a feature could look important just because it lines up with income
- source: ACS 2014-18 median household income (table B19013), joined on `geoid` (built by `src/build_acs.py`)
  - same ACS release the SLD used for car ownership → same households, same years, same block-group boundaries

**Fallback:** if `acs_blockgroups.csv.gz` hasn't been built yet, use the SLD's `low_wage_share` (share of residents' jobs paying ≤ $1,250/month, LEHD 2017), flipped so higher = richer.
This is a much weaker income measure. Treat figures made with it as provisional.

In [23]:
from pathlib import Path

ACS_PATH = Path("../data/processed/acs_blockgroups.csv.gz")

if ACS_PATH.exists():
    acs = pd.read_csv(ACS_PATH, dtype={"geoid": str})
    df = df.merge(acs, on="geoid", how="left")   # one-to-one on block group
    INCOME_COL = "median_hh_income"              # higher = richer
    INCOME_SOURCE = "ACS 2014-18 median household income"
else:
    df["income_proxy"] = -df["low_wage_share"]   # flip: higher = richer
    INCOME_COL = "income_proxy"
    INCOME_SOURCE = "SLD low-wage share (income proxy, ACS not yet joined)"

print("Income measure:", INCOME_SOURCE)
print(f"Block groups missing income: {df[INCOME_COL].isna().mean():.1%} (dropped from Steps 5-6)")

Income measure: ACS 2014-18 median household income
Block groups missing income: 2.7% (dropped from Steps 5-6)


## Step 5 · Figure 3: density **and** income held fixed

- split into 3 income groups (equal-size, like density) → 3 × 3 = 9 groups of similar density and income
- re-rank each feature **inside** each of the 9 groups → percentile within group
- same profile as before: 10 bins → household-weighted % with 2+ cars
- rows = income group, columns = feature, lines = density group (same blue ramp as Fig 2)
- read it like Fig 2: steep line = feature still tracks car ownership among similar-density, similar-income neighborhoods

Design choices
- small multiples with a **shared y-axis** → slopes compare directly across all 27 lines (position on a common scale)
- density group is ordinal → single-hue light-to-dark blue, matching Fig 2
- each line averages ~1,000+ block groups per bin, so bin means are precise. The real uncertainty is the spread *within* bins
- **shaded band = middle 50% of block groups** (25th-75th percentile) in each bin → shows how much neighborhoods vary around each mean

In [24]:
DENSITY_GROUPS = ["Low density", "Medium density", "High density"]
INCOME_GROUPS = ["Lower income", "Middle income", "Higher income"]

# 3 equal-size income groups (block groups with no income value get NaN and are left out)
df["income_group"] = pd.qcut(df[INCOME_COL], 3, labels=INCOME_GROUPS)

tables = []
for inc in INCOME_GROUPS:
    for dens in DENSITY_GROUPS:
        cell = df[(df["income_group"] == inc) & (df["density_group"] == dens)]
        for col, label in FEATURES.items():
            if col == "pop_density":
                continue  # density is held fixed here, so skip it
            t = profile(cell, col, label)   # percentiles are re-ranked inside this cell
            t["income_group"] = inc
            t["density_group"] = dens
            tables.append(t)
prof3 = pd.concat(tables, ignore_index=True)

# block groups in each density x income cell
pd.crosstab(df["income_group"], df["density_group"])

density_group,Low density,Medium density,High density
income_group,,,
Lower income,9266,11692,16298
Middle income,12802,12513,11942
Higher income,15366,13206,8679


In [25]:
base3 = alt.Chart().encode(
    x=alt.X("x:Q", title="Percentile within group", scale=alt.Scale(domain=[0, 100])),
    color=alt.Color("density_group:N", title="Density group", sort=DENSITY_GROUPS,
                    scale=alt.Scale(range=["#9ecae1", "#4292c6", "#08306b"]),
                    # legend shows the line colors, not the faint band colors
                    legend=alt.Legend(symbolType="stroke", symbolStrokeWidth=3, symbolOpacity=1)),
)
# middle 50% of block groups in each bin
bands = base3.mark_area(opacity=0.15).encode(
    y=alt.Y("q25:Q", title="2+ cars (%)", scale=alt.Scale(domain=[0, 100])),
    y2="q75:Q",
)
means = base3.mark_line(point=alt.OverlayMarkDef(size=20)).encode(
    y="y:Q",
    tooltip=["feature", "income_group", "density_group", "bin", "y", "q25", "q75", "n"],
)

fig3 = alt.layer(bands, means, data=prof3).properties(
    width=190,
    height=150,
).facet(
    row=alt.Row("income_group:N", title=None, sort=INCOME_GROUPS,
                header=alt.Header(labelAngle=0, labelAlign="left", labelFontWeight="bold")),
    column=alt.Column("feature:N", title=None, sort=["Mixed use", "Street connectivity", "Transit access"],
                      header=alt.Header(labelFontWeight="bold", labelFontSize=13)),
).properties(
    title=alt.Title(
        "Holding density and income fixed, which features still track car ownership?",
        subtitle=[f"Households with 2+ cars (%), by percentile of each feature within each density × income group. "
                  f"Line = household-weighted mean; band = middle 50% of block groups.",
                  f"{df['income_group'].notna().sum():,} block groups in 54 metros of 1M+ people. "
                  f"Income: {INCOME_SOURCE}. Source: EPA Smart Location Database v3.0."],
    ),
)

fig3.save("../figures/fig3_within_density_income.png", scale_factor=2)
fig3.save("../figures/fig3_within_density_income.html")
fig3

alt.FacetChart(...)

**How to read Fig 3**

- each panel compares neighborhoods with similar density **and** similar income, so a slope can't come from those two differences
- the lines sit at different heights because denser and richer groups own different numbers of cars on average; what matters is the **slope inside each line**
- Fig 4 (next) boils the 27 lines down to one number per feature

## Step 6 · Figure 4: how much of each link survives the controls?

- one number per feature: **gap** = % with 2+ cars in the least-urban bin minus the most-urban bin
- computed 3 ways:
  - **No controls**: Fig 1
  - **Density held fixed**: Fig 2, averaged over the 3 density groups
  - **Density + income held fixed**: Fig 3, averaged over the 9 groups
- population density can't be held fixed against itself → its last point holds **income** fixed only
- a gap that shrinks a lot as controls are added was partly riding on density or income
- caveat for mixed use: its curve is an inverted U, so end-to-end gaps understate its mid-range effect
- a negative gap means the most-urban tenth owns *more* cars than the least-urban tenth

Design choices
- position along one common x-axis (the most accurate encoding for comparing magnitudes)
- control level is ordinal → light-to-dark ramp; a grey rule connects each feature's points to show how far it moves
- features sorted by raw gap, so the ranking reads top to bottom

In [26]:
def gap_table(p, group_cols):
    """Gap (least-urban bin minus most-urban bin) per feature, averaged over the groups in group_cols."""
    g = p.sort_values("bin").groupby(["feature"] + group_cols)["y"].agg(["first", "last"])
    g["gap"] = g["first"] - g["last"]
    return g.groupby("feature")["gap"].mean()

# density can only be held fixed against income: re-rank density inside each income group
tables = []
for inc in INCOME_GROUPS:
    t = profile(df[df["income_group"] == inc], "pop_density", "Population density")
    t["income_group"] = inc
    tables.append(t)
prof_density_income = pd.concat(tables, ignore_index=True)

LEVELS = ["No controls", "Density held fixed", "Density + income held fixed"]
gaps = pd.concat([
    gap_table(prof, []).rename(LEVELS[0]),
    gap_table(prof2, ["density_group"]).rename(LEVELS[1]),
    pd.concat([gap_table(prof3, ["income_group", "density_group"]),
               gap_table(prof_density_income, ["income_group"])]).rename(LEVELS[2]),
], axis=1)
gaps.round(1)

,No controls,Density held fixed,Density + income held fixed
feature,,,
Mixed use,-2.0,6.7,10.4
Population density,51.8,NaN,46.4
Street connectivity,31.2,10.9,8.7
Transit access,36.1,23.2,18.6


In [27]:
gaps_long = gaps.reset_index().melt(id_vars="feature", var_name="controls", value_name="gap").dropna()
feature_order = gaps[LEVELS[0]].sort_values(ascending=False).index.tolist()

base = alt.Chart(gaps_long).encode(
    y=alt.Y("feature:N", title=None, sort=feature_order, axis=alt.Axis(labelFontSize=12)),
)
rules = base.mark_rule(color="#bbbbbb", strokeWidth=2).encode(
    x=alt.X("min(gap):Q"), x2="max(gap):Q",
)
dots = base.mark_circle(size=140, opacity=1).encode(
    x=alt.X("gap:Q", title="Drop in % of households with 2+ cars, least → most urban tenth (points)",
            scale=alt.Scale(domain=[-5, 60])),
    color=alt.Color("controls:N", title=None, sort=LEVELS,
                    scale=alt.Scale(domain=LEVELS, range=["#c6dbef", "#4292c6", "#08306b"]),
                    legend=alt.Legend(orient="top")),
    tooltip=["feature", "controls", alt.Tooltip("gap:Q", format=".1f")],
)
zero = alt.Chart(pd.DataFrame({"x": [0]})).mark_rule(color="black", strokeWidth=1).encode(x="x:Q")
fig4 = (zero + rules + dots).properties(
    width=480,
    height=170,
    title=alt.Title(
        "How much of each feature's link to car ownership survives the controls?",
        subtitle=[f"Gaps averaged over density (3) and density × income (9) groups. Population density's last point holds income only.",
                  f"Income: {INCOME_SOURCE}. Source: EPA Smart Location Database v3.0."],
    ),
)

fig4.save("../figures/fig4_gap_summary.png", scale_factor=2)
fig4.save("../figures/fig4_gap_summary.html")
fig4

alt.LayerChart(...)

**What Fig 4 shows** (ACS 2014-18 income)

| Feature | No controls | Density fixed | Density + income fixed |
| --- | --- | --- | --- |
| Population density | 51.8 | n/a | 46.4 |
| Transit access | 36.1 | 23.2 | 18.6 |
| Mixed use | −2.0 | 6.7 | 10.4 |
| Street connectivity | 31.2 | 10.9 | 8.7 |

- **Density dominates**: even among similar-income neighborhoods, the densest tenth has ~46 points fewer households with 2+ cars than the least dense
- **Mixed use is the only feature that gets *stronger* with controls**: −2 → 6.7 → 10.4
  - with no controls, the least-mixed block groups include dense, residential-only apartment areas that already own few cars, which hides the mixed-use effect
  - once neighborhoods are compared at similar density and income, more mixed use goes with ~10 points fewer 2+ car households
- **Street connectivity mostly rode on density**: 31 → 9 points. Well-connected grids are usually dense, so the raw link overstated it
- **Transit access keeps the most** of the three non-density features (18.6), but about half its raw gap was density and income. Step 8 shows this lead mostly disappears when neighborhoods are compared within their own metro
- answer to our question: mixed use has a **real but modest** link to lower car ownership. It's comparable to street connectivity and about half of transit access, and well below density. Our hypothesis that mixed use matters *most* is not supported

Limits
- these are associations, not causal effects. Households who don't want to drive may choose mixed-use neighborhoods (self-selection)
- terciles are coarse controls. Some density and income variation remains inside each group
- block groups are small. A residential-only block group next to a shopping street counts as "unmixed" even though shops are a short walk away (modifiable areal unit problem)

## Step 7 · Anomaly check: dense but poorly connected block groups

- Fig 2 oddity: in the **high-density** group, the least-connected tenth has unusually *low* car ownership
- guess: large-block public or older apartment housing → lower income, fewer cars
- test: compare income and car ownership in that tenth vs the rest of the high-density group
- second guess: it's mostly one metro (New York), whose large blocks and car-free households pull the tenth down → check which metros it comes from

In [28]:
hd = df[df["density_group"] == "High density"].copy()
hd["conn_pct"] = hd["intersection_density"].rank(pct=True) * 100
hd["tenth"] = np.where(hd["conn_pct"] <= 10, "Least-connected tenth", "Other 90%")

# show income in its original units (dollars for ACS, low-wage share for the proxy)
income_shown = "median_hh_income" if INCOME_COL == "median_hh_income" else "low_wage_share"

hd.groupby("tenth").agg(
    block_groups=("geoid", "size"),
    pct_2plus_cars=("pct_2plus_cars", "mean"),
    pct_0_cars=("pct_0_cars", "mean"),
    median_income_measure=(income_shown, "median"),
    share_in_nyc_metro=("cbsa_name", lambda s: s.str.startswith("New York").mean()),
    share_lower_income_group=("income_group", lambda s: (s == "Lower income").mean()),
).round(2)

,block_groups,pct_2plus_cars,pct_0_cars,median_income_measure,share_in_nyc_metro,share_lower_income_group
tenth,,,,,,
Least-connected tenth,3828,27.42,35.51,53100.0,0.47,0.47
Other 90%,34459,42.48,19.16,57386.0,0.21,0.42


In [29]:
# which metros do those least-connected, high-density block groups sit in?
hd[hd["tenth"] == "Least-connected tenth"]["cbsa_name"].value_counts().head(8)

cbsa_name
New York-Newark-Jersey City, NY-NJ-PA           1789
Los Angeles-Long Beach-Anaheim, CA               385
Chicago-Naperville-Elgin, IL-IN-WI               176
Washington-Arlington-Alexandria, DC-VA-MD-WV     165
Miami-Fort Lauderdale-Pompano Beach, FL          162
Dallas-Fort Worth-Arlington, TX                  120
San Diego-Chula Vista-Carlsbad, CA                76
San Francisco-Oakland-Berkeley, CA                69
Name: count, dtype: int64

**What Step 7 shows**

| High-density block groups | Least-connected tenth | Other 90% |
| --- | --- | --- |
| Households with 2+ cars | 27% | 42% |
| Households with 0 cars | 36% | 19% |
| Median household income | $53,100 | $57,386 |
| Share in lower-income third | 47% | 42% |
| Share in New York metro | 47% | 21% |

- **income explains only a little**: the least-connected tenth is somewhat poorer (about $4,000 lower median income), not enough for a 15-point drop in 2+ car ownership
- **New York explains most of it**: nearly half these block groups are in the NYC metro, more than twice the rate in the rest of the group. NYC combines very low car ownership with large blocks (e.g. tower-in-the-park housing on superblocks), so "least connected" here largely means "in New York"
- implication: connectivity's dip at the low end of the high-density line is a metro-composition effect, not evidence that poor connectivity reduces car ownership
- possible robustness check: rank features **within each metro** instead of nationally, so one metro can't dominate a group

## Step 8 · Robustness: do the Fig 4 results hold up?

We decided on two checks **before** seeing their results, and report them whatever they show.

1. **Within-metro percentiles.** Steps 1-6 rank every block group against the whole country, so a metro like New York
   (dense, transit-rich, few cars) can fill an entire bin (Step 7). Here each feature is ranked **inside its own metro**,
   and the density and income thirds are also cut inside each metro. A block group is now "most urban" relative to its own region.
2. **A different outcome.** Car ownership is one measure of car dependence. Here we also use the ACS share of commuters who
   **drive alone** to work (table B08301), weighted by number of workers.

Same method as Fig 4 otherwise: 10 bins, gap = least-urban bin minus most-urban bin, averaged over the control groups.

In [30]:
def gaps_for(data, outcome, weight, within_metro, features=None):
    """Fig 4's three gap columns for one outcome, using national or within-metro percentiles."""
    d = data.copy()
    rank_within = ["cbsa"] if within_metro else None

    if within_metro:
        # cut density and income into thirds inside each metro (0, 1, 2 -> names)
        d["density_group"] = d.groupby("cbsa")["pop_density"].transform(
            lambda s: pd.qcut(s, 3, labels=False)).map(dict(enumerate(DENSITY_GROUPS)))
        d["income_group"] = d.groupby("cbsa")[INCOME_COL].transform(
            lambda s: pd.qcut(s, 3, labels=False)).map(dict(enumerate(INCOME_GROUPS)))

    def prof_for(subset, col, label):
        return profile(subset, col, label, outcome=outcome, weight=weight, rank_within=rank_within)

    no_ctrl, dens_ctrl, dens_inc_ctrl, density_by_income = [], [], [], []
    for col, label in (features or FEATURES).items():
        no_ctrl.append(prof_for(d, col, label))

        if col == "pop_density":
            # density can only be held fixed against income
            for inc in INCOME_GROUPS:
                t = prof_for(d[d["income_group"] == inc], col, label)
                t["income_group"] = inc
                density_by_income.append(t)
            continue

        for dens in DENSITY_GROUPS:
            t = prof_for(d[d["density_group"] == dens], col, label)
            t["density_group"] = dens
            dens_ctrl.append(t)
            for inc in INCOME_GROUPS:
                cell = d[(d["density_group"] == dens) & (d["income_group"] == inc)]
                t = prof_for(cell, col, label)
                t["density_group"] = dens
                t["income_group"] = inc
                dens_inc_ctrl.append(t)

    return pd.concat([
        gap_table(pd.concat(no_ctrl), []).rename(LEVELS[0]),
        gap_table(pd.concat(dens_ctrl), ["density_group"]).rename(LEVELS[1]),
        pd.concat([gap_table(pd.concat(dens_inc_ctrl), ["income_group", "density_group"]),
                   gap_table(pd.concat(density_by_income), ["income_group"])]).rename(LEVELS[2]),
    ], axis=1)


# 2 outcomes x 2 ranking methods = 4 versions of the Fig 4 table
OUTCOMES = {
    "pct_2plus_cars": ("Households with 2+ cars", "households"),
    "pct_drove_alone": ("Commuters driving alone", "workers"),
}
RANKINGS = {False: "National percentiles", True: "Within-metro percentiles"}

robust = []
for outcome, (outcome_label, weight) in OUTCOMES.items():
    for within_metro, ranking_label in RANKINGS.items():
        t = gaps_for(df, outcome, weight, within_metro)
        t["outcome"] = outcome_label
        t["ranking"] = ranking_label
        robust.append(t.reset_index())
robust = pd.concat(robust, ignore_index=True)

# one row per feature, one column per version, at the strictest control level
robust.pivot_table(index="feature", columns=["outcome", "ranking"], values=LEVELS[2]).round(1)

outcome             Commuters driving alone                           \
ranking                National percentiles Within-metro percentiles   
feature                                                                
Mixed use                               7.7                      4.9   
Population density                     42.2                     20.9   
Street connectivity                     1.7                      5.6   
Transit access                         16.2                      6.5   

outcome             Households with 2+ cars                           
ranking                National percentiles Within-metro percentiles  
feature                                                               
Mixed use                              10.4                     10.2  
Population density                     46.4                     29.3  
Street connectivity                     8.7                     11.5  
Transit access                         18.6                     11.1

In [31]:
# full table: all three control levels for every version
robust.set_index(["outcome", "ranking", "feature"])[LEVELS].round(1)

No controls  \
outcome                 ranking                  feature                            
Households with 2+ cars National percentiles     Mixed use                   -2.0   
                                                 Population density          51.8   
                                                 Street connectivity         31.2   
                                                 Transit access              36.1   
                        Within-metro percentiles Mixed use                    1.3   
                                                 Population density          40.9   
                                                 Street connectivity         29.3   
                                                 Transit access              32.0   
Commuters driving alone National percentiles     Mixed use                    0.2   
                                                 Population density          43.7   
                                                 Street connectivity         18.1   
                                                 Transit access              28.9   
                        Within-metro percentiles Mixed use                   -0.5   
                                                 Population density          23.6   
                                                 Street connectivity         13.5   
                                                 Transit access              17.5   

                                                                      Density held fixed  \
outcome                 ranking                  feature                                   
Households with 2+ cars National percentiles     Mixed use                           6.7   
                                                 Population density                  NaN   
                                                 Street connectivity                10.9   
                                                 Transit access                     23.2   
                        Within-metro percentiles Mixed use                          11.0   
                                                 Population density                  NaN   
                                                 Street connectivity                12.7   
                                                 Transit access                     20.2   
Commuters driving alone National percentiles     Mixed use                           7.0   
                                                 Population density                  NaN   
                                                 Street connectivity                 2.4   
                                                 Transit access                     16.3   
                        Within-metro percentiles Mixed use                           5.5   
                                                 Population density                  NaN   
                                                 Street connectivity                 4.9   
                                                 Transit access                      9.4   

                                                                      Density + income held fixed  
outcome                 ranking                  feature                                           
Households with 2+ cars National percentiles     Mixed use                                   10.4  
                                                 Population density                          46.4  
                                                 Street connectivity                          8.7  
                                                 Transit access                              18.6  
                        Within-metro percentiles Mixed use                                   10.2  
                                                 Population density                          29.3  
                                                 Street connectivity              

### Figure 5: the strictest-control gap under each check

- same encoding as Fig 4: position on one shared x-axis, features in the same order
- color = ranking method (nominal → two distinct hues); grey rule = how far a feature moves between the two
- one panel per outcome, sharing the x-axis so the two outcomes compare directly

In [32]:
fig5_data = robust[["feature", "outcome", "ranking", LEVELS[2]]].rename(columns={LEVELS[2]: "gap"})
RANK_COLORS = alt.Scale(domain=list(RANKINGS.values()), range=["#08306b", "#e08214"])

base5 = alt.Chart().encode(
    y=alt.Y("feature:N", title=None, sort=feature_order, axis=alt.Axis(labelFontSize=12)),
)
rules5 = base5.mark_rule(color="#bbbbbb", strokeWidth=2).encode(x="min(gap):Q", x2="max(gap):Q")
dots5 = base5.mark_circle(size=140, opacity=1).encode(
    x=alt.X("gap:Q", title="Drop from least → most urban tenth (points)", scale=alt.Scale(domain=[-5, 50])),
    color=alt.Color("ranking:N", title=None, scale=RANK_COLORS, legend=alt.Legend(orient="top")),
    tooltip=["feature", "outcome", "ranking", alt.Tooltip("gap:Q", format=".1f")],
)
zero5 = alt.Chart().mark_rule(color="black", strokeWidth=1).encode(x=alt.datum(0))

fig5 = alt.layer(zero5, rules5, dots5, data=fig5_data).properties(
    width=300, height=170,
).facet(
    column=alt.Column("outcome:N", title=None, sort=list(o for o, _ in OUTCOMES.values()),
                      header=alt.Header(labelFontWeight="bold", labelFontSize=13)),
).properties(
    title=alt.Title(
        "Do the results hold with within-metro ranking and a commuting outcome?",
        subtitle=["Gap with density and income held fixed (population density: income only), averaged over groups.",
                  f"Income and commute mode: ACS 2014-18. Source: EPA Smart Location Database v3.0."],
    ),
)

fig5.save("../figures/fig5_robustness.png", scale_factor=2)
fig5.save("../figures/fig5_robustness.html")
fig5

alt.FacetChart(...)

**What Step 8 shows** (gap with density and income held fixed, in points)

| Feature | 2+ cars, national | 2+ cars, within metro | Drive alone, national | Drive alone, within metro |
| --- | --- | --- | --- | --- |
| Population density | 46.4 | 29.3 | 42.2 | 20.9 |
| Mixed use | 10.4 | 10.2 | 7.7 | 4.9 |
| Transit access | 18.6 | 11.1 | 16.2 | 6.5 |
| Street connectivity | 8.7 | 11.5 | 1.7 | 5.6 |

- **Density dominates in every version.** It shrinks within metros (part of its national gap is New York vs. everywhere else), but it is still 2-4× any other feature
- **Mixed use is the most stable result.** ~10 points for car ownership whether ranked nationally or within metros, and it gets stronger with controls in all four versions. It is smaller for commuting (5-8 points), which makes sense: a shop or job mix nearby changes errands more than the commute
- **Transit's second place does not hold up.** Ranked within metros, transit (11.1) is level with mixed use (10.2) and connectivity (11.5). Much of transit's national lead came from comparing transit-rich metros with car-oriented ones, not neighborhoods within the same metro
- **Street connectivity is fragile.** Between 2 and 12 points depending on the version, and close to zero for driving alone nationally
- updated answer: **density first by a wide margin; mixed use, transit, and connectivity each have a modest link of similar size once neighborhoods are compared within their own metro.** The mixed-use hypothesis is still not supported as "matters most," but mixed use's modest link is the most consistent of the three

Caveats
- within-metro thirds mean "dense for this metro," so a "high density" block group in Phoenix is not as dense as one in New York. This is a different question (relative position in your region), not a strictly better one
- drive-alone share only covers workers, and the ACS commute question asks about last week's usual mode

## Step 9 · Commuting in and out: what is mixed use actually measuring?

**Why this check.** In our proposal review, Prof. Kale pointed out a mismatch in our two measures. Mixed use (`D2A_EPHHM`) is built from
the **jobs located** in a block group, while car ownership describes the **people who live** there. Jobs in a busy area are often filled by
people who commute in from elsewhere (his example: restaurant workers downtown who can't afford to live downtown). If so, a high mixed-use
score may say little about the residents whose cars we are counting.

**What we added.** `src/build_sld.py` now keeps two LEHD 2017 counts from the SLD: `jobs` (jobs located in the block group) and
`resident_workers` (working people who live there). Their ratio, **jobs per resident worker**, shows which way commuters must flow:

- well below 1: more working residents than local jobs, so most residents **commute out**
- around 1: jobs and working residents are **roughly balanced**
- well above 1: more jobs than working residents, so the area **draws commuters in**

**Groups.** We cut the ratio at 0.5 and 2, a factor of two either side of balance (symmetric on a log scale). We chose these cutoffs
before computing any gaps, and Step 9b repeats the check with equal-size thirds instead.

**Test.** Re-run Fig 4's strictest comparison (density and income held fixed) **inside each commuting group**. If the mixed-use link
holds within groups, it is not just a by-product of where jobs sit relative to residents. If it disappears, mixed use and
jobs-to-residents balance cannot be told apart in this data.

**Limit of the data.** The ratio is a *net balance*, not a count of actual trips. The SLD cannot say where any individual commutes.
Tracking real home-to-work flows would need the Census LEHD Origin-Destination files (LODES).

In [33]:
ROLES = ["Residents commute out", "Roughly balanced", "Draws commuters in"]
df["commute_role"] = pd.cut(df["jobs_per_resident_worker"], [-np.inf, 0.5, 2, np.inf], labels=ROLES, right=False)

print(f"Block groups with no resident workers (ratio undefined, left out of Step 9): "
      f"{df['commute_role'].isna().mean():.1%}")
print(f"Rank correlation, mixed use vs jobs per resident worker: "
      f"{df['mixed_use'].corr(df['jobs_per_resident_worker'], method='spearman'):.2f}")

# what each group looks like
df.groupby("commute_role", observed=True).agg(
    block_groups=("geoid", "size"),
    median_jobs=("jobs", "median"),
    median_resident_workers=("resident_workers", "median"),
    median_mixed_use=("mixed_use", "median"),
    pct_2plus_cars=("pct_2plus_cars", "mean"),
    median_income=("median_hh_income", "median"),
    median_low_wage_gap=("low_wage_gap", "median"),   # job low-wage share minus resident low-wage share
).round(2)

Block groups with no resident workers (ratio undefined, left out of Step 9): 1.4%
Rank correlation, mixed use vs jobs per resident worker: 0.88


,block_groups,median_jobs,median_resident_workers,median_mixed_use,pct_2plus_cars,median_income,median_low_wage_gap
commute_role,,,,,,,
Residents commute out,71658,92.0,634.0,0.38,57.62,68750.0,0.09
Roughly balanced,29141,567.0,623.0,0.70,53.52,65833.0,0.05
Draws commuters in,12460,2389.5,575.0,0.75,47.81,62679.0,-0.02


In [34]:
# Fig 4's gaps, computed separately inside each commuting group (national percentiles, 2+ cars outcome)
commute_gaps = []
for role in ROLES:
    t = gaps_for(df[df["commute_role"] == role], "pct_2plus_cars", "households", within_metro=False)
    t["group"] = role
    commute_gaps.append(t.reset_index())
t = gaps.reset_index()          # Fig 4's table: all block groups
t["group"] = "All block groups"
commute_gaps.append(t)
commute_gaps = pd.concat(commute_gaps, ignore_index=True)

commute_gaps.pivot_table(index="feature", columns="group", values=LEVELS[2]).reindex(index=feature_order, columns=["All block groups"] + ROLES).round(1)

group,All block groups,Residents commute out,Roughly balanced,Draws commuters in
feature,,,,
Population density,46.4,47.6,47.0,43.3
Transit access,18.6,15.6,15.3,19.0
Street connectivity,8.7,6.3,4.9,7.6
Mixed use,10.4,-1.9,-3.1,6.5


### Step 9b · Is the result an artifact of our choices?

Three variations on the mixed-use row, each at the strictest control level:

1. **Equal-size thirds** of jobs per resident worker instead of the 0.5 / 2 cutoffs
2. **Within-metro percentiles** (as in Step 8)
3. **Drive-alone share** as the outcome (as in Step 8)

We also treat jobs per resident worker as a **feature of its own** and compute its Fig 4 gap, to see how its link compares with mixed use's.

In [35]:
rows = []

def mixed_use_gaps(groups, label, **kw):
    for g in groups.cat.categories:
        t = gaps_for(df[groups == g], **kw)
        rows.append({"check": label, "group": g, "mixed_use_gap": t.loc["Mixed use", LEVELS[2]]})

thirds = pd.qcut(df["jobs_per_resident_worker"], 3, labels=["Lowest third", "Middle third", "Highest third"])
mixed_use_gaps(df["commute_role"], "Main check (cutoffs 0.5 and 2)",
               outcome="pct_2plus_cars", weight="households", within_metro=False)
mixed_use_gaps(thirds, "Equal-size thirds", outcome="pct_2plus_cars", weight="households", within_metro=False)
mixed_use_gaps(df["commute_role"], "Within-metro percentiles",
               outcome="pct_2plus_cars", weight="households", within_metro=True)
mixed_use_gaps(df["commute_role"], "Outcome: drive alone",
               outcome="pct_drove_alone", weight="workers", within_metro=False)

sensitivity = pd.DataFrame(rows)
sensitivity["group"] = sensitivity.groupby("check").cumcount().map({0: "Low ratio", 1: "Middle", 2: "High ratio"})
print("Mixed-use gap with density + income held fixed, inside each jobs-per-resident-worker group (points)")
sensitivity.pivot(index="check", columns="group", values="mixed_use_gap").reindex(
    index=sensitivity["check"].unique(), columns=["Low ratio", "Middle", "High ratio"]).round(1)

Mixed-use gap with density + income held fixed, inside each jobs-per-resident-worker group (points)


group,Low ratio,Middle,High ratio
check,,,
Main check (cutoffs 0.5 and 2),-1.9,-3.1,6.5
Equal-size thirds,-8.0,-8.3,2.8
Within-metro percentiles,-1.7,-1.4,5.1
Outcome: drive alone,2.8,-1.7,-4.3


In [36]:
# jobs per resident worker as a feature in its own right, next to mixed use and density
JOBS_FEATURES = {
    "jobs_per_resident_worker": "Jobs per resident worker",
    "mixed_use": "Mixed use",
    "pop_density": "Population density",
}
pd.concat({
    "National percentiles": gaps_for(df, "pct_2plus_cars", "households", False, features=JOBS_FEATURES),
    "Within-metro percentiles": gaps_for(df, "pct_2plus_cars", "households", True, features=JOBS_FEATURES),
}).round(1)

No controls  \
                         feature                                 
National percentiles     Jobs per resident worker          8.8   
                         Mixed use                        -2.0   
                         Population density               51.8   
Within-metro percentiles Jobs per resident worker          9.7   
                         Mixed use                         1.3   
                         Population density               40.9   

                                                   Density held fixed  \
                         feature                                        
National percentiles     Jobs per resident worker                17.4   
                         Mixed use                                6.7   
                         Population density                       NaN   
Within-metro percentiles Jobs per resident worker                18.0   
                         Mixed use                               11.0   
                         Population density                       NaN   

                                                   Density + income held fixed  
                         feature                                                
National percentiles     Jobs per resident worker                         18.0  
                         Mixed use                                        10.4  
                         Population density                               46.4  
Within-metro percentiles Jobs per resident worker                         16.2  
                         Mixed use                                        10.2  
                         Population density                               29.3

### Figure 6: each feature's gap inside each commuting group

- same encoding as Figs 4 and 5: position on one shared x-axis, features in the same order, strictest control level
- commuting group is ordinal (out → balanced → in), so it gets a single-hue light-to-dark ramp. Green keeps it distinct from the blue
  control-level ramp in Fig 4
- the black tick marks Fig 4's all-block-group value, so the shift caused by splitting into commuting groups reads directly as distance
  from the tick
- grey rule = range across the three groups

In [37]:
fig6_data = commute_gaps[["feature", "group", LEVELS[2]]].rename(columns={LEVELS[2]: "gap"})
by_role = fig6_data[fig6_data["group"] != "All block groups"]
overall = fig6_data[fig6_data["group"] == "All block groups"]

y_enc = alt.Y("feature:N", title=None, sort=feature_order, axis=alt.Axis(labelFontSize=12))
x_scale = alt.Scale(domain=[-10, 55])

rules6 = alt.Chart(by_role).mark_rule(color="#bbbbbb", strokeWidth=2).encode(
    y=y_enc, x=alt.X("min(gap):Q", scale=x_scale), x2="max(gap):Q",
)
dots6 = alt.Chart(by_role).mark_circle(size=140, opacity=1).encode(
    y=y_enc,
    x=alt.X("gap:Q", scale=x_scale,
            title="Drop in % of households with 2+ cars, least → most urban tenth (points)"),
    color=alt.Color("group:N", title=None, sort=ROLES,
                    scale=alt.Scale(domain=ROLES, range=["#a1d99b", "#41ab5d", "#00441b"]),
                    legend=alt.Legend(orient="top")),
    tooltip=["feature", "group", alt.Tooltip("gap:Q", format=".1f")],
)
ticks6 = alt.Chart(overall).mark_tick(color="black", thickness=3, size=22).encode(
    y=y_enc, x=alt.X("gap:Q", scale=x_scale),
    tooltip=["feature", alt.Tooltip("gap:Q", format=".1f", title="All block groups (Fig 4)")],
)
zero6 = alt.Chart(pd.DataFrame({"x": [0]})).mark_rule(color="black", strokeWidth=1).encode(x="x:Q")

fig6 = (zero6 + rules6 + ticks6 + dots6).properties(
    width=480,
    height=170,
    title=alt.Title(
        "Does each feature's link survive holding commuting balance fixed?",
        subtitle=["Gap with density and income held fixed (population density: income only), inside each commuting group.",
                  "Black tick = all block groups (Fig 4). Groups: jobs per resident worker below 0.5, 0.5-2, above 2.",
                  "Source: EPA Smart Location Database v3.0 (LEHD 2017 jobs and workers); income: ACS 2014-18."],
    ),
)

fig6.save("../figures/fig6_commuting_balance.png", scale_factor=2)
fig6.save("../figures/fig6_commuting_balance.html")
fig6

alt.LayerChart(...)

**What Step 9 shows** (gap with density and income held fixed, in points)

| Feature | All block groups (Fig 4) | Residents commute out | Roughly balanced | Draws commuters in |
| --- | --- | --- | --- | --- |
| Population density | 46.4 | 47.6 | 47.0 | 43.3 |
| Transit access | 18.6 | 15.6 | 15.3 | 19.0 |
| Mixed use | 10.4 | −1.9 | −3.1 | 6.5 |
| Street connectivity | 8.7 | 6.3 | 4.9 | 7.6 |

- **Density, transit, and connectivity barely move.** Their links to car ownership look the same whether a neighborhood exports workers,
  is balanced, or imports them.
- **Mixed use's ~10-point link disappears** in the commute-out and balanced groups, which together hold 89% of block groups.
  It survives only in job centers (6.5 points).
- **Why: our mixed-use score mostly measures jobs relative to residents.** Mixed use and jobs per resident worker have a rank correlation
  of 0.88. `D2A_EPHHM` scores the balance between households and eight job types, and in a typical block group jobs are scarce
  (median 0.30 jobs per resident worker). So a higher mixed-use score mostly means *more jobs located among the residents*. Once that
  balance is held fixed, the part of mixed use that is left (the variety of job types) has almost no link to car ownership.
- **Jobs per resident worker is a stronger predictor than mixed use.** Treated as a feature of its own, it has an 18.0-point gap with
  density and income held fixed (16.2 within metros). Mixed use has 10.4. That ties transit for second place nationally, behind density.
- **The result is not an artifact of our choices (Step 9b).** With equal-size thirds, within-metro ranking, or the drive-alone outcome,
  the within-group mixed-use gap stays between −8 and +7 points and never approaches the overall 10.
- **Prof. Kale's restaurant example is not the typical pattern in job centers.** There, the median job is about as likely to be low-wage as
  the median resident's job (gap −0.02). It is in residential block groups that the few local jobs are lower-paid than residents' own jobs (+0.09).

**What this means for RQ**

- Our mixed-use finding is better described as a **jobs-near-residents** finding: at similar density and income, neighborhoods with more
  jobs relative to their working residents have fewer households with 2+ cars.
- The data cannot tell apart two explanations:
  1. residents of job-rich neighborhoods can work and run errands nearby, so fewer of them need two cars (a jobs-housing balance version of our hypothesis)
  2. job-rich neighborhoods differ in ways we haven't measured that also lower car ownership (scarce or costly parking, older street layouts),
     and their jobs are mostly held by people who commute in (Prof. Kale's concern)
- Separating them would take **LEHD Origin-Destination (LODES)** data, which records home and work block for each job. With it we could
  measure what share of residents actually work in or near their own neighborhood.

Caveats
- when two measures are this closely correlated, holding one fixed leaves little independent variation in the other. "No link within groups"
  means this data cannot separate them, not proof that the variety of job types is irrelevant
- jobs per resident worker is a net balance. A balanced block group could still have every resident commuting out and every job filled from outside
- LEHD counts are from 2017 and car ownership from ACS 2014-18, so the two are close in time but not identical

## Next steps

- **Maps:** bivariate choropleth (car ownership × on-road CO₂ per km²) for two contrasting metros